# K5 · Đánh giá toàn diện M1–M4, M* + RAG (T13)
Sinh trên test_synth / test_gold / test_adv cho từng adapter, closed-book và + RAG; fact-EM giá, tỷ lệ bịa giá, từ chối, ROUGE-L, độ trễ; paired bootstrap so với baseline; LLM-as-judge (Gemini) + hiệu chỉnh với người chấm.

Add Data: dataset repo + output của K4 (thư mục `runs/`) hoặc dùng adapter trên HF Hub.

In [ ]:
%%capture
!pip install -q "transformers==4.56.2" "trl==0.21.0" "peft==0.17.0" "bitsandbytes==0.47.0" "accelerate>=1.4" "datasets>=3.0" rank-bm25 optuna
!pip install -q google-genai

In [ ]:
import sys, glob, json, os, time
sys.path.insert(0, glob.glob('/kaggle/input/**/code/src', recursive=True)[0])
from kimochi.kaggle_env import setup, gpu_info
ROOT = setup()
import pandas as pd
from pathlib import Path
RES = ROOT / 'results'
print(gpu_info())

In [ ]:
from kimochi import prompts, metrics, llm, retriever
ADAPTERS = {Path(p).parent.name: str(Path(p).parent / 'adapter') for p in glob.glob('/kaggle/input/**/runs/*/result.json', recursive=True)
            if not Path(p).parent.name.startswith('lc_')}
# hoặc: ADAPTERS = {'qwen_nf4_s42': '<user>/kimochi-qwen-nf4-s42', ...}
print(ADAPTERS)
EVAL_SETS = ['test_synth'] + [s for s in ['test_gold', 'test_adv'] if (ROOT / f'data/splits/{s}.jsonl').exists()]
R = retriever.HybridRetriever(retriever.load_docs()); docs_by_id = {d['doc_id']: d for d in R.docs}
ctx = lambda q: [docs_by_id[i] for i in R.search(q, 'bm25_filter', k=3)]
GEN = RES / 'gen'; GEN.mkdir(exist_ok=True)

In [ ]:
rows = []
for run, ad in ADAPTERS.items():
    mk, q = run.split('_')[:2]
    model, tok, _ = llm.load(mk, q, adapter=ad)
    for es in EVAL_SETS:
        ev = prompts.load_eval_set(es)
        for mode in ['closed', 'rag']:
            msgs = [prompts.build_messages(x, 'rag' if mode == 'rag' else 'zero', docs=ctx(x) if mode == 'rag' else None) for x in ev['query']]
            preds, lat = llm.generate(model, tok, msgs, bs=8)
            out = ev.assign(pred=preds, latency_batch_s=lat, system=f'{run}_{mode}', eval_set=es)
            out.to_json(GEN / f'{run}_{mode}_{es}.jsonl', orient='records', lines=True, force_ascii=False)
            rows.append({'system': f'{run}_{mode}', 'eval_set': es, **metrics.summarize(metrics.score_generations(out))}); print(rows[-1])
    q30 = prompts.load_eval_set('test_synth').head(30)['query']
    lat1 = llm.latency_single(model, tok, [prompts.build_messages(x, 'rag', docs=ctx(x)) for x in q30])
    rows.append({'system': f'{run}_rag', 'eval_set': 'latency30', 'latency_p50': float(pd.Series(lat1).median()),
                 'latency_p95': float(pd.Series(lat1).quantile(.95))})
    del model, tok; llm.free()
ft = pd.DataFrame(rows); ft.to_csv(RES / 'finetuned_eval.csv', index=False); ft.round(3)

## Khoảng tin cậy và so sánh ghép cặp (thay cho K-Fold)

In [ ]:
def load_gen(system, es):
    f = {**{p.stem: p for p in GEN.glob('*.jsonl')}}
    name = next(k for k in f if k.endswith(es) and k.startswith(system))
    return metrics.score_generations(pd.read_json(f[name], lines=True))

ES = 'test_gold' if 'test_gold' in EVAL_SETS else 'test_synth'
cmp = []
systems = [p.stem.rsplit('_' + ES, 1)[0] for p in GEN.glob(f'*_{ES}.jsonl')]
for s in systems:
    d = load_gen(s, ES)
    m, lo, hi = metrics.bootstrap_ci(d['rougeL'])
    cmp.append({'system': s, 'rougeL': m, 'ci_low': lo, 'ci_high': hi, 'fact_em': d['fact_ok'].mean()})
cmp = pd.DataFrame(cmp).sort_values('rougeL', ascending=False); cmp.to_csv(RES / f'ci_{ES}.csv', index=False); display(cmp.round(3))
best, ref = cmp.system.iloc[0], 'qwen_rag'  # so với baseline B2 cùng họ model
if any(s.startswith(ref) for s in systems):
    a, b = load_gen(best, ES), load_gen(ref, ES)
    print(best, 'vs', ref, metrics.paired_bootstrap(a['rougeL'].values, b['rougeL'].values))

## LLM-as-judge (Gemini) và hiệu chỉnh với người chấm
Người chấm điền `data/splits/human_scores.csv` (cột `id`, `system`, `human_score` 1–5) cho ~50 mẫu; báo cáo tương quan Spearman giữa judge và người để chứng minh judge đáng tin.

In [ ]:
JUDGE_N = 150
from scipy.stats import spearmanr
jud = []
for s in cmp.system.head(3).tolist() + [x for x in systems if x in ('qwen_zero', 'qwen_rag', 'llama_rag')]:
    d = load_gen(s, ES).head(JUDGE_N)
    for r in d.itertuples():
        sc, why = metrics.judge(r.query, r.reference, r.pred)
        jud.append({'id': r.id, 'system': s, 'judge': sc, 'reason': why}); time.sleep(0.5)
jud = pd.DataFrame(jud); jud.to_csv(RES / 'judge_scores.csv', index=False)
display(jud.groupby('system').judge.agg(['mean', 'count']).round(2))
hs = ROOT / 'data/splits/human_scores.csv'
if hs.exists():
    h = pd.read_csv(hs).merge(jud, on=['id', 'system'])
    print('Spearman judge vs người:', spearmanr(h.human_score, h.judge), 'n =', len(h))

## Mười ca lỗi điển hình (Phần 5 — phân tích lỗi)

In [ ]:
worst = load_gen(cmp.system.iloc[0], ES).sort_values('rougeL').head(10)[['query', 'reference', 'pred', 'intent']]
worst.to_csv(RES / 'error_cases.csv', index=False); worst